1.phase

In [ ]:
from cProfile import label

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/snli")
print(dataset)
print(dataset["train"][0])

# output =  train, validation, test

train = [
    {hypothesis: "A person on a horse jumps over a broken down airplane." ,
     premise: "A person is outdoors, on a horse." ,
     label: "entailment"},
    {hypothesis: "A person on a horse jumps over a broken down airplane." ,
     premise: "A person is outdoors, on a horse." , 
    label: "entailment"},


]

1.1 MODEL loading

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"  # Change this to your desired model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

1.2 tokenzie data and load into model

In [ ]:
# prompting format
dataset = [
    'hypothesis: "....". Premise: " ". label""',
    'hypothesis: "....". Premise: " ". label""',
    'hypothesis: "....". Premise: " ". label""',
           ]

# tokenize the dataset 
tokenized_dataset = tokenizer(dataset, padding=True, truncation=True, return_tensors="pt")

In [ ]:
# this cell loads model, loads the lora adapter and trains the model on the dataset via the lora adapter
# at the end of this cell you have a finetuned model which you can use for inference on the test set

# monitor the loss of the model and save it into a .txt file 
# and at the end visualize it in a graph, (x= iteration, y= loss)
# must be done by llm: introduce validation and visualize validation accuracy, 
# --> early stopping. --> model accuracy on training shall not exceed validation accuracy
# if reached that point stop the training and save the model weights in the folder where we saved loss and accuracy values in .txt file, and visualize it in a graph

# # Install required libraries if not already installed
!pip install transformers datasets peft accelerate --quiet

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Set model and dataset names
model_name = "gpt2"  # Change to your LLM

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],  # Adjust for your model architecture
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)

# Load and tokenize dataset (example with wikitext)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:10000]")  # Replace with your dataset

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # Will be overridden by max_steps
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    max_steps=10000,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Start training
trainer.train()

# save in the folder 

1.3 test lora model performance on test data



In [ ]:
# load finetuned models and test dataset
# same prompt, same tokenziation, same loading to model 
# make alist of correct label, prompt, and model output for each test example, and save it in a .txt file /excel 
# confusion matrix, precision, recall, f1 score, accuracy, etc.

2.phase: bonus task


In [ ]:
# load the finedtuned model via huggingface 


prompt = "you got a task whihc you coudl solve like this example: ----- .the possible labels are... . only one word which is the label.  now solve this new task: ...."

# this new prompt, same tokenziation, same loading to model(load finedtuned model)
 
